# Risk & factor analytics with `Performance`

**Purpose:** Use one `Performance` object to compute return, risk, tail, benchmark-relative, rolling-regression, and factor-model analytics.

**Prerequisites:** `03_analytics/performance_analytics.ipynb`.

**What you'll learn:**

- Build a small fund/benchmark return panel from the shared synthetic fixture.
- Inspect benchmark regression and risk diagnostics.
- Read per-period returns back out of `Performance` with `returns()` / `returns_for_ticker()`.
- Connect return-based factor analytics to later portfolio factor-sensitivity notebooks.

Every analytic in this notebook flows through a single `Performance` instance: construct it once from prices or returns, then call methods for the views you need.


## Build a small panel

We simulate two correlated daily return series — a fund and a benchmark — over one trading year, then construct a single `Performance` for the panel.

The synthetic walk comes from `_shared.random_walk_panel`, the example-suite fixture generator: it owns the business-day calendar and the seeded RNG so every notebook draws its toy data the same way. The fund is deliberately built as a **single-index** series (`fund = β·bench + residual`) so the regressions further down have something to recover.

Note the constructor: `Performance.from_panel` with a `"returns"` `PerformancePanel` takes per-period returns directly. There is no reason to compound returns into a price path in Python only to have Rust difference them again.


In [1]:
import sys
sys.path.insert(0, "..")

from _shared import random_walk_panel

from finstack_quant.analytics import Performance, PerformancePanel
from finstack_quant.core.dates import FiscalConfig

# `_shared.random_walk_panel` supplies the business-day axis (pd.bdate_range) and a
# single deterministic seeded stream. Two independent driver walks: the benchmark
# itself, and an idiosyncratic series that becomes the fund's residual.
BETA = 0.88
panel = random_walk_panel(
    253,  # 253 prices -> 252 return observations, i.e. one trading year
    2,
    seed=42,
    drift=[0.0004, 0.00025],  # benchmark drift, fund alpha
    vol=[0.01, 0.0035],  # benchmark vol, residual vol
    names=["BENCH", "IDIO"],
)
bench_returns, idio_returns = panel.returns

# Single-index fixture: fund return = BETA * benchmark + idiosyncratic residual.
fund_returns = [BETA * b + e for b, e in zip(bench_returns, idio_returns, strict=True)]

# Hand the raw returns to Rust rather than compounding prices in Python; the
# complete actual valuation grid includes the initial price date and all endpoints.
perf = Performance.from_panel(PerformancePanel(dates=panel.dates, series=[fund_returns, bench_returns], kind="returns", ticker_names=["FUND", "BENCH"], benchmark_ticker="BENCH", frequency="daily"))
print("Tickers:", perf.get_ticker_names())
print("Bench idx:", perf.get_benchmark_idx())
print("Active dates:", len(perf.get_dates()))
print(f"Designed beta: {BETA:.2f} (the regressions below should recover roughly this)")


Tickers: ['FUND', 'BENCH']
Bench idx: 1
Active dates: 252
Designed beta: 0.88 (the regressions below should recover roughly this)


## Single-factor benchmark regression

`Performance.beta()` returns OLS beta with standard error and a 95% CI per ticker; `greeks()` returns annualized alpha, beta, and R². Both use the panel's designated benchmark.


In [2]:
betas = perf.beta()
gks = perf.greeks(risk_free_rate=0.0)

fund_beta = betas[0]
fund_gr = gks[0]
print(f"Beta: {fund_beta.beta:.4f} (SE {fund_beta.std_err:.4f})")
print(f"95% CI: [{fund_beta.ci_lower:.4f}, {fund_beta.ci_upper:.4f}]")
print(f"Greeks alpha (ann.): {fund_gr.alpha:.6f}")
print(f"Greeks beta: {fund_gr.beta:.4f}; R²: {fund_gr.r_squared:.4f}")

Beta: 0.8730 (SE 0.0230)
95% CI: [0.8277, 0.9184]
Greeks alpha (ann.): 0.045708
Greeks beta: 0.8730; R²: 0.8519


## Rolling greeks

`Performance.rolling_greeks(ticker_idx, window, risk_free_rate=0.0)` fits the single-index model on a moving window and returns time series of alpha and beta aligned to right-labelled window-end dates.


In [3]:
rg = perf.rolling_greeks(0, window=63, risk_free_rate=0.0)
print(f"Rolling windows: {len(rg.alphas)}")
print(f"Last alpha: {rg.alphas[-1]:.6f}; last beta: {rg.betas[-1]:.4f}")

Rolling windows: 190
Last alpha: -0.109975; last beta: 0.8733


## VaR, ES, and tail diagnostics

`Performance` exposes historical, parametric, and Cornish–Fisher VaR plus expected shortfall, as well as higher moments (`skewness`, `kurtosis`) and the asymmetric `tail_ratio`.


In [4]:
conf = 0.95
print(f"Historical VaR ({conf:.0%}): {perf.value_at_risk(conf).iloc[0]:.6f}")
print(f"Parametric VaR ({conf:.0%}): {perf.parametric_var(conf).iloc[0]:.6f}")
from finstack_quant.analytics import AnalyticsError
try:
    print(f"Cornish–Fisher VaR ({conf:.0%}): {perf.cornish_fisher_var(conf).iloc[0]:.6f}")
except AnalyticsError as error:
    print("Cornish–Fisher estimate rejected:", error)
print(f"Expected shortfall ({conf:.0%}): {perf.expected_shortfall(conf).iloc[0]:.6f}")
print(f"Skewness: {perf.skewness().iloc[0]:.4f}")
print(f"Kurtosis: {perf.kurtosis().iloc[0]:.4f}")
print(f"Tail ratio: {perf.tail_ratio(confidence=0.95).iloc[0]:.4f}")

Historical VaR (95%): -0.013842
Parametric VaR (95%): -0.013762
Cornish–Fisher VaR (95%): -0.014054
Expected shortfall (95%): -0.018323
Skewness: -0.1342
Kurtosis: 0.3035
Tail ratio: 1.2378


## Benchmark-relative metrics

Up/down capture, capture ratio, tracking error, information ratio, R², and batting average — all per-ticker against the panel benchmark.


In [5]:
print(f"Up capture: {perf.up_capture().iloc[0]:.4f}")
print(f"Down capture: {perf.down_capture().iloc[0]:.4f}")
print(f"Capture ratio: {perf.capture_ratio().iloc[0]:.4f}")
print(f"Tracking error: {perf.tracking_error().iloc[0]:.6f}")
print(f"Information ratio: {perf.information_ratio().iloc[0]:.4f}")
print(f"R-squared: {perf.r_squared().iloc[0]:.4f}")
print(f"Batting average: {perf.batting_average().iloc[0]:.4f}")

Up capture: 0.7414
Down capture: 0.9264
Capture ratio: 0.8003
Tracking error: 0.059722
Information ratio: 0.0047
R-squared: 0.8519
Batting average: 0.4762


## Drawdown episodes

`drawdown_details(ticker_idx, n=...)` returns the deepest drawdown episodes for a single ticker with start, valley, and recovery dates.


In [6]:
print(f"Max drawdown (fund): {perf.max_drawdown().iloc[0]:.2%}")
episodes = perf.drawdown_details(0, n=3)
for i, ep in enumerate(episodes, start=1):
    end = ep.end
    end_s = end.isoformat() if end else "(ongoing)"
    print(
        f"  #{i} Start: {ep.start.isoformat()}, Valley: {ep.valley.isoformat()}, "
        f"End: {end_s}, Depth: {ep.max_drawdown:.2%}"
    )

Max drawdown (fund): -6.33%
  #1 Start: 2024-01-18, Valley: 2024-02-19, End: 2024-03-25, Depth: -6.33%
  #2 Start: 2024-11-20, Valley: 2024-12-05, End: (ongoing), Depth: -5.68%
  #3 Start: 2024-04-11, Valley: 2024-04-29, End: 2024-05-08, Depth: -4.24%


## Multi-factor model

`Performance.multi_factor_greeks(ticker_idx, factor_returns, risk_free_rate=0.0)` fits a multivariate OLS against several aligned factor return series and returns alpha, betas, R², adjusted R², and residual volatility.

The market factor here is the panel's own benchmark, so take its returns from `Performance.get_returns_for_ticker(idx)` — the canonical per-period return accessor (`get_returns()` gives the whole panel). Do **not** rebuild them by un-compounding `cumulative_returns()`: that reintroduces return-linking arithmetic in Python, and any disagreement with the Rust convention shows up silently as a wrong beta.


In [7]:
import random

active_dates = perf.get_dates()
n_active = len(active_dates)
random.seed(7)
smb_returns = [random.gauss(0.0, 0.005) for _ in range(n_active)]
hml_returns = [random.gauss(0.0, 0.004) for _ in range(n_active)]

# The market factor is simply the benchmark's own per-period returns. Take them
# from the canonical accessor -- never re-derive them by un-compounding
# `cumulative_returns()`, and no longer via the `excess_returns([0.0] * n)` trick.
market_returns = perf.get_returns_for_ticker(perf.get_benchmark_idx())

mf = perf.multi_factor_greeks(0, [market_returns, smb_returns, hml_returns], risk_free_rate=0.0)
print(f"Alpha (ann.): {mf.alpha:.6f}")
print(f"Betas [mkt, size, value]: {[round(b, 4) for b in mf.betas]}")
print(f"R²: {mf.r_squared:.4f}; Adj R²: {mf.adjusted_r_squared:.4f}")
print(f"Residual vol: {mf.residual_vol:.6f}")

# `get_returns()` is the panel-shaped sibling: one series per ticker, same convention.
panel_returns = perf.get_returns()
print(f"\nget_returns() panel: {len(panel_returns)} series x {len(panel_returns[perf.get_benchmark_idx()])} periods")

Alpha (ann.): 0.049927
Betas [mkt, size, value]: [np.float64(0.8724), np.float64(-0.0631), np.float64(0.0253)]
R²: 0.8532; Adj R²: 0.8515
Residual vol: 0.056472

get_returns() panel: 2 series x 252 periods


## Periodic returns

`lookback_returns` always reports period-to-date returns for MTD / QTD / YTD / FYTD. FYTD uses a January-1 fiscal start by default or the supplied `FiscalConfig`. `rolling_returns(ticker_idx, window)` compounds returns over a rolling window for trailing 1Y / 3Y views.


In [8]:
ref = perf.get_dates()[-1]
lb = perf.lookback_returns(ref, FiscalConfig(4, 1))
print(f"As of {ref.isoformat()}:")
print(f"  MTD: {lb.mtd[0]:.4f}")
print(f"  QTD: {lb.qtd[0]:.4f}")
print(f"  YTD: {lb.ytd[0]:.4f}")
print(f"  FYTD (Apr): {lb.fytd[0]:.4f}")

roll_1y = perf.rolling_returns(0, window=252)
print(f"Trailing 1Y rolling returns: {len(roll_1y.values)} windows")
if len(roll_1y.values) > 0:
    print(f"  Last window end: {roll_1y.dates[-1].isoformat()}")
    print(f"  Last 1Y compounded return: {roll_1y.values[-1]:.4f}")

As of 2024-12-18:
  MTD: -0.0034
  QTD: 0.0506
  YTD: 0.4149
  FYTD (Apr): 0.3619
Trailing 1Y rolling returns: 1 windows
  Last window end: 2024-12-18
  Last 1Y compounded return: 0.4149


## Takeaways

- `Performance` is the single binding entry point: construct from prices (or returns) and reach every analytic as a method.
- Benchmark regressions (`beta`, `greeks`, `rolling_greeks`) and multi-factor models (`multi_factor_greeks`) cover the alpha / beta / R² questions.
- Need the raw per-period returns back? Use `returns()` (panel) or `returns_for_ticker(idx)` (single series). Un-compounding `cumulative_returns()` in Python is return-linking logic that belongs in Rust.
- Tail risk (`value_at_risk`, `expected_shortfall`, parametric / Cornish–Fisher variants, higher moments, `tail_ratio`) and benchmark-relative metrics live on the same instance.
- Periodic and rolling views (`lookback_returns`, `rolling_returns`, `rolling_sharpe`, `rolling_volatility`) answer MTD / QTD / YTD / FYTD and trailing-window questions directly.


## Analyst program: aligned synthetic factor diagnostics

This 504-day proxy panel is separate from the three-period native book return history.
Returns are daily decimals; VaR/ES are daily return loss fractions, not USD. Multiplying
by a disclosed capital amount yields a proxy USD risk measure. Square-root-of-time scaling
requires independent, stable daily increments and does not model liquidity or jumps.


In [9]:
from datetime import date

import math
import numpy as np
from _shared.analyst_history import risk_panel
from finstack_quant.analytics import Performance
from finstack_quant.core.dates import FiscalConfig
panel = risk_panel()
assert panel.index.is_unique and not panel.isna().any().any()
performance = Performance.from_returns(panel[["portfolio", "benchmark"]], frequency="daily", benchmark_ticker="benchmark", initial_date=date(2023, 2, 7))
confidence = 0.95
measures = {"historical_var":float(performance.value_at_risk(confidence).iloc[0]), "normal_var":float(performance.parametric_var(confidence).iloc[0]), "historical_es":float(performance.expected_shortfall(confidence).iloc[0])}
from finstack_quant.analytics import AnalyticsError
try:
    measures["cornish_fisher_var"] = float(performance.cornish_fisher_var(confidence).iloc[0])
except AnalyticsError as error:
    print("Cornish–Fisher estimate rejected:", error)
assert all(math.isfinite(value) for value in measures.values())
assert abs(measures["historical_es"]) >= abs(measures["historical_var"])
factors = panel[["market", "rates", "credit"]].to_numpy().T.tolist()
fit = performance.multi_factor_greeks(0, factors, risk_free_rate=0.0)
design = np.column_stack([np.ones(len(panel)), panel[["market", "rates", "credit"]]])
ols = np.linalg.lstsq(design, panel["portfolio"].to_numpy(), rcond=None)[0]
assert np.allclose(fit.betas, ols[1:], atol=1e-10), "Dimensionless OLS loadings, independent NumPy reference"
print("Daily return-risk measures:",measures)
print("Factor loadings [market,rates,credit]:",fit.betas,"R-squared:",fit.r_squared)
print("Incorrect 95% one-day VaR times sqrt(10) (not 99%):",measures["normal_var"]*math.sqrt(10))


Cornish–Fisher estimate rejected: Validation error: Cornish–Fisher quantile curve is not globally nondecreasing
Daily return-risk measures: {'historical_var': -0.006019362790807607, 'normal_var': -0.0064614185165724316, 'historical_es': -0.007306500424087516}
Factor loadings [market,rates,credit]: [0.40599196 0.43211163 0.29236851] R-squared: 0.9685047556760439
Incorrect 95% one-day VaR times sqrt(10) (not 99%): -0.02043279942795531


## Analyst lesson 3.2 — Audit the tail estimator and factor covariance


In [10]:
from finstack_quant.core.math import linalg

returns = panel["portfolio"].to_numpy()
ordered = np.sort(returns)
h = (len(ordered)-1) * (1-confidence)
k = int(math.floor(h))
manual_var = ordered[k] + (h-k) * (ordered[k+1]-ordered[k])
manual_es = ordered[ordered <= manual_var].mean()
mass = (1 - confidence) * len(ordered)
whole = min(int(math.floor(mass)), len(ordered))
closed_tail = float(ordered[0] if whole == 0 else ordered[:whole].sum() / mass + (0 if whole >= len(ordered) else ordered[whole] * ((mass - whole) / mass)))
assert abs(manual_var-measures["historical_var"]) < 1e-12
assert abs(closed_tail-measures["historical_es"]) < 1e-12
design = np.column_stack((np.ones(len(returns)), panel[["market", "rates", "credit"]].to_numpy()))
ols = np.linalg.lstsq(design, returns, rcond=None)[0]
residual = returns - design.dot(ols)
assert abs(float(np.mean(residual))) < 1e-9, float(np.mean(residual))
assert np.allclose(design.T @ residual, 0, atol=1e-10)
manual_r_squared = 1 - float(residual @ residual) / float((returns-returns.mean()) @ (returns-returns.mean()))
assert abs(manual_r_squared-fit.r_squared) < 1e-10
factor_observations = panel[["market","rates","credit"]].to_numpy()
shrunk, intensity = linalg.ledoit_wolf_shrinkage(factor_observations)
shrunk = np.asarray(shrunk)
assert 0 <= intensity <= 1 and np.linalg.eigvalsh(shrunk).min() > 0
print(f"Manual R-7 lower quantile={manual_var:.8f}; closed-tail mean={closed_tail:.8f}")
print(f"Daily residual volatility={residual.std(ddof=1):.8f}; Ledoit-Wolf intensity={intensity:.6f}")
print("Shrunk daily factor covariance:\n", shrunk)

single_beta = performance.beta()[0].beta
benchmark_returns = panel["benchmark"].to_numpy()
manual_beta = np.cov(returns, benchmark_returns, ddof=1)[0,1] / benchmark_returns.var(ddof=1)
assert abs(single_beta-manual_beta) < 1e-12
print(f"Benchmark beta={single_beta:.8f}; independent covariance ratio={manual_beta:.8f}")


Manual R-7 lower quantile=-0.00601936; closed-tail mean=-0.00730650
Daily residual volatility=0.00074045; Ledoit-Wolf intensity=0.008823
Shrunk daily factor covariance:
 [[ 8.51806482e-05 -5.05361180e-07  2.44019898e-06]
 [-5.05361180e-07  7.17057841e-06 -1.50254377e-07]
 [ 2.44019898e-06 -1.50254377e-07  1.25076473e-05]]
Benchmark beta=1.27975892; independent covariance ratio=1.27975892


## Analyst lesson 3.2 — Compare a moment correction with the empirical tail


In [11]:
from datetime import date

import pandas as pd

rng = np.random.default_rng(73)
normal_draws = rng.normal(0, 0.01, 504)
skewed_returns = normal_draws - 0.002 * ((normal_draws/0.01)**2 - 1)
skewed_performance = Performance.from_returns(pd.DataFrame({"skewed":skewed_returns},index=panel.index),frequency="daily", initial_date=date(2023, 2, 7))
skewed_measures = {
    "historical":float(skewed_performance.value_at_risk(0.95).iloc[0]),
    "Gaussian":float(skewed_performance.parametric_var(0.95).iloc[0]),
}
try:
    skewed_measures["Cornish-Fisher"] = float(skewed_performance.cornish_fisher_var(0.95).iloc[0])
except AnalyticsError as error:
    print("Cornish–Fisher estimate rejected:", error)
assert all(math.isfinite(value) for value in skewed_measures.values())
assert "Cornish-Fisher" not in skewed_measures or abs(skewed_measures["Cornish-Fisher"]-skewed_measures["Gaussian"]) > 1e-5
print("Controlled skewed daily returns:",skewed_measures)
print("A correction changes the model; closeness to one empirical quantile is not a guarantee out of sample.")


Cornish–Fisher estimate rejected: Validation error: Cornish–Fisher quantile curve is not globally nondecreasing
Controlled skewed daily returns: {'historical': -0.01888169803503662, 'Gaussian': -0.015824040408918803}
A correction changes the model; closeness to one empirical quantile is not a guarantee out of sample.


## Analyst lesson 3.2 — Discrete VaR and exact tail-mass ES


In [12]:
from datetime import date

# One hundred equally likely dollar-loss states; four A defaults and four different B defaults.
loss_a = np.zeros(100); loss_a[:4] = 0.1
loss_b = np.zeros(100); loss_b[4:8] = 0.1
state_returns = pd.DataFrame({"A":-loss_a,"B":-loss_b,"AplusB":-(loss_a+loss_b)},index=pd.bdate_range("2025-01-01",periods=100))
discrete = Performance.from_returns(state_returns,frequency="daily", initial_date=date(2024, 12, 31))
var_values = discrete.value_at_risk(0.95)
assert var_values["A"] == var_values["B"] == 0 and var_values["AplusB"] == -0.1
# Exact 95% ES averages exactly five worst equiprobable states, including partial threshold mass if needed.
exact_es_a = np.sort(loss_a)[-5:].mean()
exact_es_b = np.sort(loss_b)[-5:].mean()
exact_es_sum = np.sort(loss_a+loss_b)[-5:].mean()
assert exact_es_sum <= exact_es_a+exact_es_b
print("Loss-positive VaR:",(-var_values).to_dict())
print("Exact tail-mass ES:",{"A":exact_es_a,"B":exact_es_b,"AplusB":exact_es_sum})
print("API exact fractional empirical ES:",(-discrete.expected_shortfall(0.95)).to_dict())
print("The API integrates exactly the requested empirical tail mass, including fractional observations; tied thresholds do not add extra mass.")


Loss-positive VaR: {'A': -0.0, 'B': -0.0, 'AplusB': 0.1}
Exact tail-mass ES: {'A': np.float64(0.08), 'B': np.float64(0.08), 'AplusB': np.float64(0.1)}
API exact fractional empirical ES: {'A': 0.07999999999999993, 'B': 0.07999999999999993, 'AplusB': 0.1}
The API integrates exactly the requested empirical tail mass, including fractional observations; tied thresholds do not add extra mass.


## Analyst lesson 3.2 — Change both confidence and horizon


In [13]:
from statistics import NormalDist

mu = returns.mean(); sigma = returns.std(ddof=1)
z99 = NormalDist().inv_cdf(0.99)
manual_ten_day = 10 * mu - z99 * sigma * math.sqrt(10)
native_ten_day = float(performance.parametric_var(0.99,horizon_periods=10).iloc[0])
assert abs(manual_ten_day-native_ten_day) < 1e-9
naive = measures["normal_var"] * math.sqrt(10)
assert abs(naive-native_ten_day) > 1e-4
print(f"99%/10-period Gaussian return VaR={native_ten_day:.6%}")
print(f"Incorrect 95%/1-period sqrt(10) shortcut={naive:.6%}")
print("Assumptions: independent, stationary Gaussian increments, unchanged positions, additive-return approximation.")


99%/10-period Gaussian return VaR=-2.668011%
Incorrect 95%/1-period sqrt(10) shortcut=-2.043280%
Assumptions: independent, stationary Gaussian increments, unchanged positions, additive-return approximation.


## Analyst lesson 3.2 — Prepare an exposure-neutral signal


In [14]:
from finstack_quant.features import neutralize_and_zscore

raw_signal = [1.0,2.0,2.0,4.0]
market_exposure = [0.0,1.0,0.0,1.0]
scores = np.asarray(neutralize_and_zscore(raw_signal,["2025-01-15"]*4,[market_exposure]))
assert abs(scores.mean()) < 1e-12
assert abs(scores @ np.asarray(market_exposure)) < 1e-12
assert abs(scores.std(ddof=0)-1) < 1e-12
print("Aligned neutralized population-z-scores:",scores)
print("Neutrality removes this linear exposure in this cross-section; it is not a forecast of alpha.")


Aligned neutralized population-z-scores: [-0.63245553 -1.26491106  0.63245553  1.26491106]
Neutrality removes this linear exposure in this cross-section; it is not a forecast of alpha.


## Analyst lesson 3.2 — Recover normal ES from VaR assumptions


In [15]:
from finstack_quant.models.factor.risk import position_risk_report
from statistics import NormalDist

alpha, daily_sigma = 0.95, 0.01
z = NormalDist().inv_cdf(alpha)
phi = math.exp(-0.5*z*z)/math.sqrt(2*math.pi)
expected_normal_es = -daily_sigma * phi/(1-alpha)
es_reference = position_risk_report({"position_ids": ["unit_exposure"], "confidence": alpha, "source": {"kind": "parametric", "weights": [1.0], "covariance": [[daily_sigma**2]]}}).decomposition
assert abs(es_reference.portfolio_es-expected_normal_es) < 1e-9
print(f"Zero-mean normal VaR={es_reference.portfolio_var:.6%}; ES={es_reference.portfolio_es:.6%}")
print(f"ES/VaR magnitude ratio={phi/((1-alpha)*z):.6f}; this ratio depends on confidence and distribution.")


Zero-mean normal VaR=-1.644854%; ES=-2.062713%
ES/VaR magnitude ratio=1.254040; this ratio depends on confidence and distribution.
